# Lezione 22 — Uno schema esplicito per la memoria

**Come si usa questo notebook.** Come sempre. Prerequisiti: Lezione 1
(campi critici vs recuperabili) e Lezione 21 (fine della Fase 3). Apre la
**Fase 4** (rappresentare le memorie): schema, tempo, importanza, entita',
grafo, retrieval ibrido, contraddizioni.

Finora ogni lezione ha lavorato su colonne di un DataFrame, ricalcolate da
zero quando servivano (embedding, similarita', cluster...). Da qui in poi
le lezioni **aggiungono informazione alla stessa memoria** (recency,
importanza composita, entita', posizione nel grafo): serve un contratto
esplicito su cosa e' *sempre* vero di un record, prima di costruirci sopra.

**Cosa saprai fare alla fine:** definire uno schema esplicito per una
memoria con `dataclasses` (libreria standard, nessuna dipendenza nuova),
scrivere una funzione di validazione che distingue errori bloccanti da
avvisi, e capire perche' formalizzare un contratto conta quando piu'
lezioni costruiscono sopra agli stessi dati.

## Parte 1 — Teoria: un contratto, non solo delle colonne

La Lezione 1 ha gia' distinto campi **critici** (`memory_id`, `text`,
`timestamp` — senza uno di questi la riga perde identita', si scarta) da
campi **recuperabili** (`type`, `importance` — si imputano con un flag).
Quella distinzione era gia' uno **schema implicito**: oggi lo rendiamo
esplicito e eseguibile, con due strumenti della libreria standard di
Python:

- **`@dataclass`** — una classe che dichiara i campi di un record con i
  loro tipi, senza scrivere a mano `__init__`. Non valida nulla da sola
  (Python non controlla i tipi a runtime): e' solo una struttura dati
  dichiarativa, piu' leggibile di un dizionario libero perche' i campi
  attesi sono scritti una volta sola, in un posto solo.
- **una funzione di validazione dedicata** — perche' la validazione vera
  (tipo corretto, range valido, valore in un insieme noto) va scritta
  esplicitamente: `dataclass` non la fa per te. La funzione restituisce
  una lista di **errori** (bloccanti: la riga non e' una memoria valida)
  separata da una lista di **avvisi** (non bloccanti: la riga e' valida ma
  qualcosa merita attenzione) — la stessa distinzione critico/recuperabile
  della Lezione 1, applicata non piu' a "scarta o imputa" ma a "rifiuta o
  accetta con nota".

Le regole, derivate da cinque lezioni di pulizia (1-5) piu' il dominio
della memoria:

| campo | regola | bloccante? |
|---|---|---|
| `memory_id` | non vuoto | si' |
| `text` | non vuoto dopo `strip()` | si' |
| `timestamp` | stringa ISO valida (`pd.to_datetime` non fallisce) | si' |
| `type` | in `{episodic, semantic, preference, unknown}` | si' |
| `importance` | numero in `[0, 1]` | no — avviso se fuori range, non blocca |

Perche' `importance` fuori range e' solo un avviso e non un blocco: un
punteggio leggermente fuori scala (es. `1.05` per un errore di
arrotondamento a monte) non toglie identita' alla memoria, a differenza di
un `memory_id` vuoto — e' esattamente la stessa logica critico/recuperabile
della Lezione 1, applicata qui a un controllo invece che a una scelta tra
scarto e imputazione.

In [1]:
from dataclasses import dataclass
import pandas as pd

TIPI_VALIDI = {'episodic', 'semantic', 'preference', 'unknown'}


@dataclass
class Memoria:
    memory_id: str
    text: str
    timestamp: str
    type: str
    importance: float


def valida_memoria(m: Memoria) -> tuple[list[str], list[str]]:
    errori, avvisi = [], []

    if not m.memory_id or not str(m.memory_id).strip():
        errori.append('memory_id vuoto')
    if not m.text or not str(m.text).strip():
        errori.append('text vuoto')
    try:
        pd.to_datetime(m.timestamp)
    except (ValueError, TypeError):
        errori.append(f'timestamp non valido: {m.timestamp!r}')
    if m.type not in TIPI_VALIDI:
        errori.append(f'type sconosciuto: {m.type!r} (attesi: {sorted(TIPI_VALIDI)})')
    if pd.isna(m.importance):
        errori.append('importance mancante')
    elif not (0.0 <= m.importance <= 1.0):
        avvisi.append(f'importance fuori range [0,1]: {m.importance}')

    return errori, avvisi


buona = Memoria('mem_001', 'Marco visited Glasgow.', '2026-07-03', 'episodic', 0.72)
print('memoria valida:', valida_memoria(buona))

rotta = Memoria('mem_002', '  ', '2026-13-40', 'urgent', 1.4)
print('memoria rotta :', valida_memoria(rotta))

memoria valida: ([], [])
memoria rotta : (['text vuoto', "timestamp non valido: '2026-13-40'", "type sconosciuto: 'urgent' (attesi: ['episodic', 'preference', 'semantic', 'unknown'])"], ['importance fuori range [0,1]: 1.4'])


Leggi l'output: la memoria "buona" non produce ne' errori ne' avvisi
(entrambe le liste vuote). La memoria "rotta" accumula **quattro** errori
in un colpo solo (testo vuoto, timestamp impossibile — mese 13, giorno
40 — type sconosciuto) e un avviso (`importance` a `1.4`, fuori range ma
non bloccante secondo la regola sopra). La funzione non si ferma al primo
problema: raccoglie **tutti** gli errori insieme, cosi' chi corregge il
record li vede tutti in un colpo solo invece di scoprirli uno alla volta a
ogni nuovo tentativo.

## Parte 2 — Esercizio guidato

Il tuo compito: costruisci una `Memoria` con `importance = 0.5` (valido) ma
`memory_id = ''` (vuoto) e verifica che `valida_memoria` produca **esattamente
un** errore, nessun avviso.

In [2]:
# Scrivi qui: crea una Memoria con memory_id vuoto e gli altri campi validi,
# chiama valida_memoria e stampa errori/avvisi.

pass

### Soluzione spiegata

Con solo `memory_id` vuoto e tutti gli altri campi validi (`text` non
vuoto, `timestamp` valido, `type` noto, `importance` in range), la
funzione deve produrre **un solo** errore (`memory_id vuoto`) e **nessun**
avviso — ogni regola e' indipendente dalle altre, controlla esattamente
la condizione che dichiara di controllare.

In [3]:
solo_id_vuoto = Memoria('', 'Testo valido.', '2026-07-03', 'episodic', 0.5)
errori, avvisi = valida_memoria(solo_id_vuoto)
print('errori:', errori)
print('avvisi:', avvisi)
assert errori == ['memory_id vuoto']
assert avvisi == []

errori: ['memory_id vuoto']
avvisi: []


## Parte 3 — Il progetto: Memory AI Lab, passo 22 — validare l'intero archivio

Applichiamo `valida_memoria` a **tutte** le memorie di train/val/test
(Lezioni 1-5): se la pulizia delle prime lezioni ha fatto il suo lavoro,
ci aspettiamo **zero errori** — la validazione qui non e' un secondo giro
di pulizia, e' una **prova di correttezza** dello schema su dati gia'
puliti, che da ora in poi ogni lezione della Fase 4 puo' dare per
scontata.

In [4]:
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_{n}.csv') for n in ['train', 'val', 'test']}

report = {}
for nome, df in insiemi.items():
    totale_errori, totale_avvisi = 0, 0
    for _, riga in df.iterrows():
        m = Memoria(riga['memory_id'], riga['text'], riga['timestamp'],
                    riga['type'], riga['importance'])
        errori, avvisi = valida_memoria(m)
        totale_errori += len(errori)
        totale_avvisi += len(avvisi)
    report[nome] = {'righe': len(df), 'errori': totale_errori, 'avvisi': totale_avvisi}

for nome, r in report.items():
    print(f"{nome:6s}: {r['righe']:3d} righe, {r['errori']} errori, {r['avvisi']} avvisi")
    assert r['errori'] == 0, f'{nome} ha memorie non valide: la pulizia delle Lezioni 1-5 e\' incompleta'

train : 213 righe, 0 errori, 0 avvisi
val   :  20 righe, 0 errori, 0 avvisi
test  :  15 righe, 0 errori, 0 avvisi


Guarda l'`assert`: non e' decorativo. Se una lezione futura (o una nuova
versione dei dati) introducesse un buco nella pulizia, questa cella si
**ferma con un errore** invece di lasciar proseguire silenziosamente le
lezioni successive su dati non validi — lo stesso principio degli
`assert` finali della Lezione 1, applicato qui all'intero archivio invece
che a una singola trasformazione.

## Cosa hai imparato

- Uno **schema esplicito** (`@dataclass` + funzione di validazione
  dedicata) rende eseguibile un contratto che prima era solo implicito nel
  codice di pulizia — utile soprattutto quando, come da qui in poi, piu'
  lezioni costruiscono sopra agli stessi record.
- La distinzione **errore bloccante / avviso non bloccante** e' la stessa
  logica critico/recuperabile della Lezione 1, applicata alla validazione
  invece che alla scelta scarto/imputazione.
- Una funzione di validazione dovrebbe raccogliere **tutti** i problemi di
  un record, non fermarsi al primo: chi corregge i dati li vede tutti
  insieme.
- Validare l'intero archivio con un `assert` finale trasforma "dovrebbe
  essere pulito" in una garanzia verificata automaticamente, non
  un'assunzione.

## Quiz

1. Perche' un `memory_id` vuoto e' un errore bloccante mentre
   un'`importance` fuori range e' solo un avviso?
2. Cosa succederebbe se `valida_memoria` si fermasse al primo errore
   trovato invece di raccogliere tutti gli errori di un record?
3. Perche' l'`assert` sul report finale (zero errori attesi su train/val/test)
   e' una verifica utile, anche se ci si aspetta che sia sempre vero?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' senza `memory_id` la riga perde identita' — non sai piu' a
   quale memoria ti riferisci, esattamente come un `reading_id` mancante
   nella Lezione 1. Un'`importance` leggermente fuori range e' comunque
   un numero collegato a una memoria identificabile: un problema di
   qualita' del dato, non di identita' del record.
2. Chi corregge i dati vedrebbe un solo problema alla volta, lo
   correggerebbe, rilancerebbe la validazione, ne troverebbe un altro, e
   cosi' via — un ciclo lento invece di una lista completa da correggere
   in un colpo solo.
3. Perche' trasforma un'assunzione implicita ("i dati puliti dovrebbero
   essere validi") in una verifica automatica: se in futuro un cambiamento
   ai dati o al codice di pulizia rompe silenziosamente questa proprieta',
   il notebook si ferma qui con un errore chiaro invece di lasciar
   proseguire le lezioni successive su dati corrotti.
</details>

## Fonti

- Python, documentazione ufficiale, modulo `dataclasses`:
  https://docs.python.org/3/library/dataclasses.html
- pandas, `to_datetime` (usato per validare il timestamp):
  https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html

## Prossima lezione

Lo schema tratta i quattro `type` come categorie intercambiabili. Non lo
sono: la prossima lezione approfondisce perche' episodic, semantic e
preference vanno trattate diversamente da qui in poi (decadimento,
importanza, aggiornamento).